# Подбор множителей V93 по прогнозу V84

Автономный ноутбук: внутри зашиты прогноз V84 и дневные компоненты текущей V93 (бакалавриат, магистратура, прочие программы). Внешние файлы проекта для запуска не нужны. Требуются `numpy`, `pandas`, `scipy`, `matplotlib`.

**Что подбирается:** по одному неотрицательному множителю для каждого потока V93. Коэффициенты оптимизируются так, чтобы сумма потоков по дням была близка к V84 в логарифмической шкале. Ограничение сохраняет общий объём V84 (около 11 400). Малый штраф удерживает множители ближе к 1.

**Ограничение интерпретации:** это калибровка V93 под V84 на тех же датах, а не независимая оценка качества и не перенос внутренних коэффициентов V84. Public score результата заранее неизвестен.

Запускайте ячейки по порядку. Результат будет сохранён отдельно как `forecast_2024_v93_v84_weighted.csv`, вместе с коэффициентами в JSON.


In [ ]:
# Если библиотек не хватает, раскомментируйте и выполните:
# %pip install numpy pandas scipy matplotlib

import base64
import hashlib
import json
import tempfile
import zipfile
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize


In [ ]:
# Встроенный архив с учительским прогнозом V84 и тремя потоками V93.
BUNDLE_B64 = (
'UEsDBBQAAAAIANMpVV2wFK1lHAIAAM8FAAAVAAAAZm9yZWNhc3RfMjAyNF92ODQuY3N2VZRLjhUxDEXnrCXP8je2l9OCZooEvX9x'
'6yHKqWGOUsm1fVI/Pr4+189fvz+/f/z5+qas/uL9Ul6WVC4aOVCWJ22TYBmoi4lnac+lL+wm0d7tQwO0KbebHydt0MD57baH5lIV'
'6urwg9ZSKxKL4iNgL+GmCmeJmxofkfLFsrYwlQpHD9UVbaSNpDLUVoSSbLa7D6C+QgN7ZYcNjeW7SaPqvhx0I2iSKpvuofmMVCvw'
'aWqV6dB+X7Ntc/pNhddWp+5KmwMFo2khLNuOvbpcjFBYyJQqtqScdLf6lCr+iCSoJoS42vPYtHGgUIlVTo2SkMVIKjC5obWs0FHG'
'6I+9vcyEUsJq4kM3cUTKzp4T4NsZCaYZaoyW9Bpqy9A7Zz6DwjuTpk7nOmgsU/TZLfk4Ad61UpZyTPfhXW9s3bKnodDukQi+1dV1'
'15hb4Jsx4ZXU3GGCPLSR/f++ukTURET1zIEK3SkEL6MG2krynXw3py4JJ0pd9tmmjcnI8R3kc0Av7oF4UBvxMvewQguIce09qrr8'
'gzzWGTWhod/leLBNFJFHFFhnSh1pcymcux6xK3Qa6O9aveLue13iKVNzFk8+eIcooqp2QGiH4uzontQzSS+MO9L9aCZcQxRm/CkO'
'KJgkRVx7B+oSPP426D4QvwQ8gJaaJPqcBTR7LBEe69DH2XnBxi/oaC70gp0Cm2YK+p5CXBtvZtcQrgqOEdq/IfwFUEsDBBQAAAAI'
'AOQ6VV01eUeL9gUAAGkOAAAYAAAAdjkzX2RlZ3JlZV9mb3JlY2FzdHMuY3N2lZfbrh23DYbv+yyyIJ4k8nGc2GmBNjXguu+fj7MD'
'a10FWb4azxY11M//oPXl84+v45fPv/7r63++fR+/f/7fj6/f+////uf3b///75fx27fvX3/l7T90qX9a+5Ou4TqPH9cca57SbXv4'
'lF3HYviZZ9WpugUy3KZk7u0UuMVeSUGK+Nbh2a+O+S1Qlq3n31883OX23nIfIjUzPaP6z1oSOYTOeCgbYjyu2PJSE3/WmG8ZOpdL'
'pY/kVOdIPiUiJttuyR6ydcrROEqJxFn9bs3ylSlDTkw9y0/emjNUYu7InXxxmqUmu8vcay/zoZo82lrn1iRv9zxZtk5vH3R0eKJL'
'qwi29OI8m55vUY3as3dfH9truOyxp1id3a+YpzPEex5bb6B8Pi0ZoT4LfLKGBjxxX93MjG1VZwTnbzh/foMi2LChg5dxWMmeyGn0'
'ZGoD6SMWmEfafvmSDa+cJyo2X9qzlqnUU7Qtg0acl7r57C1yeMcZGZnv8Qz81GG0jGdpgx4WNCK14hbBbvYPxtyT6kZhxelGt4rA'
'Az8G/rn95UybTZNWAHN3U7W3nBxncsht7GP5aCfk1pz3sKZbsaluOze4NsIBt9XmrlVhI8rBMiHtLSoOkzNPRCqnhUEwPrtINbTf'
'rYDxdKY/iwTtm09dsUBwH5TrhmSA7TjM6aErvGpF3SIZdnqrWPCykIKs+ChqTJI9M2FKmd8BiTIVmxZwWEaeaarnIM4z9ykpugfV'
'Pqa+tNfirdklDCMWx4cBjfUSD4dSdaYghbjgib+FtcTArIANYik6ibYK2ZCgmKwjGtjYszip+1YxeSBex46j/AUfQffsp4pjMXDL'
'/haW4bcKcoTOnXqsnaPxAnrkuadLQk2ApQoQ6wW5NgRHWtJLq4WMS61BqydSmLv1CPmznVtUtIIfIUzTwQFWhvdHOUqWY1e2sInj'
'2+9gSYJTPaI+/6Y7iI6zIOuTC7GCOV6mr2iTBe+gTRJoizfdA0N8DMX10CxPsQkNcGtxI/GXvqABNHN0bD0jNIcJ9dsGA6fLYWg2'
'HKO8QiUXeGweVocAM9qFO6z2x3MO4xr4ynTMuV46pC3gio1b2jOjrL0abtgH2pifoSNmUHGJShB4uy4Lm7LdqyyPUWzEdlAkrY0e'
'qt+aA48VjQE7XIFhzuZMstXko+jSXNfLgfI9rKunbUUo8H3aI2b0+BMkCA69yyZneIjbFKlQTUAo0KxMzM+fkAv9wBmmHvLsehuH'
'y/Zd9NHICrIsGAeuy2kBS4LsROWfBdlJUhNQCD0CSgP5an9h9VBWR0so+61boKPtAmtoLhZ82d3l3mrVETo7KcPuekbAAKoTkFPH'
'wijBldhe0t5hBJnEXf6OYWSHRk3FqAwxImBYexpohHSgJLPPztm8BVj4TPWwPm96kHzPtaHEgbtNfZPPL184LTlsnusVWsBBlnxc'
'BMBFnwMrrN63AG9E1kzA+3gF5YKhFLL17IgizXKfu75xFCHiSBKMfi/sIj9O8FgYM8m4J5CGwRplKAVlMT20A3W6bXbwOITgXS5v'
'AYrtIk1U8CFNUkuIEGnL2tFhLpPRn9cPWMtEsbZnnCXpjz6NlFrPgTk6PnILvAsY2ccnioPYB+cwh76GNQ9N10tF0BTrJRpJXSsZ'
'I+mKjSIhZoabcqO46/uyBUTi2uvxIfKabOJi0zc2aZTjFdK+EbS+o+/G3IIgJZdKOI2/gi1ZID+TIzsD3oK0fcT63tgcBVDZz4WY'
'pCN/x5MmwYX5Z4H2JhvVE65gQfZjfii3cI/zJA0ArNv+Y/ub217fGrhAcWPt2zPW8Fz0iAkFzqv55xdA+GpPNPiizmBJ3e3dT0+d'
'27TJXd8/AQIhMTnmJdrE3E90iT6NrRUv278nYTz+reW76S/rZPOujBtK+w/3qyjpw+LrOPZd37NtvwJ9tEp6VFseLftzoZktxnzp'
'vodrCB7VQgFCzq2TI5ngarsOLOMV/P6VgwERZ/Sz9pF6/IErCpb7ROla9cKe56oP+PAANJmBPkaRLOdKUN0rXLnL/55+/wBQSwEC'
'FAMUAAAACADTKVVdsBStZRwCAADPBQAAFQAAAAAAAAAAAAAApIEAAAAAZm9yZWNhc3RfMjAyNF92ODQuY3N2UEsBAhQDFAAAAAgA'
'5DpVXTV5R4v2BQAAaQ4AABgAAAAAAAAAAAAAAKSBTwIAAHY5M19kZWdyZWVfZm9yZWNhc3RzLmNzdlBLBQYAAAAAAgACAIkAAAB7'
'CAAAAAA='
)
_bundle = zipfile.ZipFile(BytesIO(base64.b64decode(BUNDLE_B64)))
DATA_DIR = Path(tempfile.mkdtemp(prefix="v93_v84_coefficients_"))
_bundle.extractall(DATA_DIR)

reference = pd.read_csv(DATA_DIR / "forecast_2024_v84.csv", parse_dates=["date"])
components = pd.read_csv(DATA_DIR / "v93_degree_forecasts.csv", parse_dates=["date"])
streams = ["bachelor", "master", "background"]
if not np.array_equal(reference.date, components.date):
    raise ValueError("Даты в V84 и V93 не совпадают")
if len(reference) != 73 or not np.isfinite(reference.forecast).all():
    raise ValueError("Некорректный ряд V84")
if not np.isfinite(components[streams].to_numpy(float)).all() or (components[streams].to_numpy(float) < 0).any():
    raise ValueError("Некорректные потоки V93")

X = components[streams].to_numpy(float)
y = reference.forecast.to_numpy(float)
target_volume = float(y.sum())
print(f"Дней: {len(y)}; сумма V84: {target_volume:.3f}; сумма V93 до калибровки: {X.sum():.3f}")


In [ ]:
# Настройка силы штрафа: больше значение — коэффициенты ближе к 1.
RIDGE = 0.001

def objective(weights):
    prediction = X @ weights
    log_error = np.log1p(np.maximum(prediction, 0)) - np.log1p(y)
    return float(np.mean(log_error**2) + RIDGE * np.sum((weights - 1.0)**2))

# Начинаем с общего масштабирования V93 до суммы V84; так начальная точка
# уже удовлетворяет ограничению суммы. Затем ищем три отдельных множителя.
initial = np.full(len(streams), target_volume / X.sum())
result = minimize(
    objective,
    initial,
    method="SLSQP",
    bounds=[(0.0, None)] * len(streams),
    constraints=[{"type": "eq", "fun": lambda w: float((X @ w).sum() - target_volume)}],
    options={"maxiter": 2000, "ftol": 1e-12},
)
if not result.success:
    raise RuntimeError(f"Оптимизация не сошлась: {result.message}")
weights = result.x
prediction = X @ weights

summary = pd.DataFrame({
    "stream": streams,
    "coefficient": weights,
    "original_sum": X.sum(axis=0),
    "weighted_sum": (X * weights).sum(axis=0),
})
summary


In [ ]:
def metrics(values):
    return {
        "sum": float(values.sum()),
        "MAE_vs_V84": float(np.mean(np.abs(values - y))),
        "RMSLE_vs_V84": float(np.sqrt(np.mean((np.log1p(values) - np.log1p(y))**2))),
        "balanced_MAE_vs_V84": float(np.mean(np.abs(values-y) / np.sqrt(1+y))),
    }

original = X.sum(axis=1)
print("Исходная V93:", json.dumps(metrics(original), ensure_ascii=False, indent=2))
print("После множителей:", json.dumps(metrics(prediction), ensure_ascii=False, indent=2))

plt.figure(figsize=(12, 4))
plt.plot(reference.date, y, label="V84 reference", linewidth=2)
plt.plot(reference.date, original, label="V93 before", alpha=0.7)
plt.plot(reference.date, prediction, label="V93 weighted", linewidth=1.8)
plt.legend()
plt.ylabel("Прогноз за день")
plt.title("V93 после подбора потоковых множителей по V84")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Записываем отдельный submission-файл и описание выбранных коэффициентов.
output = pd.DataFrame({"date": components.date.dt.strftime("%Y-%m-%d"), "forecast": prediction})
csv_path = Path("forecast_2024_v93_v84_weighted.csv").resolve()
json_path = Path("forecast_2024_v93_v84_coefficients.json").resolve()
output.to_csv(csv_path, index=False, float_format="%.6f")
metadata = {
    "method": "nonnegative_stream_multipliers_fit_to_v84",
    "reference_role": "teacher forecast for coefficient selection",
    "independent_validation": False,
    "streams": dict(zip(streams, map(float, weights))),
    "ridge_to_one": RIDGE,
    "sum_constraint": target_volume,
    "metrics_vs_v84": metrics(prediction),
    "reference_public_score": 0.39140,
    "reference_score_source": "user-reported; not recomputed here",
    "output_sha256": hashlib.sha256(csv_path.read_bytes()).hexdigest(),
}
json_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"CSV: {csv_path}")
print(f"Коэффициенты и происхождение: {json_path}")


## Интерпретация

Коэффициент `1.0` оставляет поток V93 без изменений. Значение выше 1 увеличивает его, ниже 1 уменьшает. Общая сумма итогового CSV принудительно совпадает с суммой V84. Метрика `RMSLE_vs_V84` измеряет только сходство с V84 на этих 73 датах; её нельзя трактовать как Public RMSLE нового файла.

Чтобы сильнее ограничить подгонку, увеличьте `RIDGE` и повторно выполните ячейки оптимизации и ниже.
